# 07B — Final LR / RF / XGBoost Spatial Benchmark

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

This notebook establishes the formal benchmarking methodology requested by the supervisor / second marker.

## Algorithms

1. Logistic Regression (LR)
2. Random Forest (RF)
3. XGBoost (XGB)

## Fair-comparison rules

All three models use:

- the same 582 static-model samples
- the same 194 landslide positives
- the same 388 pseudo-absences
- the same environmental predictors
- the same spatial blocks
- the same 5 outer validation folds

## Validation design

### Outer loop — 5-fold spatial validation

Used only for final held-out performance.

### Inner loop — 3-fold spatial validation

Used only inside each outer training set for hyperparameter tuning.

This is a nested spatial cross-validation design.

No outer test fold is used for:
- scaling
- imputation
- hyperparameter tuning
- model selection

## Primary benchmark metrics

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- PR-AUC
- Confusion matrix

Training and testing performance are both recorded to assess overfitting.

---

# Important note on feature selection

The primary benchmark deliberately uses the **same fixed environmental feature set** for LR, RF and XGBoost.

This is more appropriate for a fair algorithm comparison.

If feature selection is tested later, it must be implemented inside the inner training folds and reported as a separate experiment.


In [4]:
%pip install xgboost scikit-learn pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Run once if XGBoost is not installed:
# %pip install xgboost scikit-learn pandas numpy matplotlib

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    StratifiedGroupKFold,
    RandomizedSearchCV,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)


pandas: 3.0.2


## 1. Project paths

In [6]:
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

INPUT_PATH = (
    DEV_DIR
    / "04_model_ready"
    / "static_model_1to2.csv"
)

OUTPUT_DIR = (
    DEV_DIR
    / "05_model_results"
    / "final_LR_RF_XGB_benchmark"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(INPUT_PATH)

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_DIR)


Input: C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to2.csv
Output: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\final_LR_RF_XGB_benchmark


## 2. Load the identical benchmark dataset

In [7]:
df = pd.read_csv(INPUT_PATH)

FEATURES = [
    "elevation_m",
    "slope_deg",
    "aspect_sin",
    "aspect_cos",
    "soil_clay",
    "soil_sand",
    "soil_silt",
    "soil_bulk_density",
    "distance_major_fault_m",
]

TARGET = "target"
GROUP = "spatial_block_id"

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()
groups = df[GROUP].astype(str).copy()

print("Rows:", len(df))
print("Positives:", int((y == 1).sum()))
print("Pseudo-absences:", int((y == 0).sum()))
print("Positive prevalence:", round(float(y.mean()), 4))
print("Spatial blocks:", groups.nunique())

if len(df) != 582:
    print("WARNING: previous Stage 06 dataset had 582 rows.")

display(
    pd.DataFrame({
        "feature": FEATURES,
        "missing": [
            int(X[c].isna().sum())
            for c in FEATURES
        ]
    })
)


Rows: 582
Positives: 194
Pseudo-absences: 388
Positive prevalence: 0.3333
Spatial blocks: 167


,feature,missing
0,elevation_m,0
1,slope_deg,0
2,aspect_sin,0
3,aspect_cos,0
4,soil_clay,0
5,soil_sand,0
6,soil_silt,0
7,soil_bulk_density,0
8,distance_major_fault_m,0


## 3. Reference baselines

Because the dataset contains twice as many pseudo-absences as positives:

- positive prevalence ≈ 0.333
- an always-negative classifier would have accuracy ≈ 0.667
- a no-skill PR-AUC baseline is approximately the positive prevalence

Therefore accuracy alone is not enough.


In [8]:
positive_prevalence = float(y.mean())
majority_accuracy = float((y == 0).mean())
no_skill_pr_auc = positive_prevalence

reference_baseline = pd.DataFrame({
    "reference": [
        "Positive prevalence",
        "Always-negative accuracy",
        "No-skill PR-AUC baseline",
        "Chance balanced accuracy",
        "Chance ROC-AUC",
    ],
    "value": [
        positive_prevalence,
        majority_accuracy,
        no_skill_pr_auc,
        0.5,
        0.5,
    ]
})

display(reference_baseline)

reference_baseline.to_csv(
    OUTPUT_DIR / "metric_reference_baselines.csv",
    index=False
)


,reference,value
0,Positive prevalence,0.333333
1,Always-negative accuracy,0.666667
2,No-skill PR-AUC baseline,0.333333
3,Chance balanced accuracy,0.500000
4,Chance ROC-AUC,0.500000


# PART A — FIXED OUTER SPATIAL FOLDS

## 4. Create one shared 5-fold outer split

These exact outer folds are reused for LR, RF and XGBoost.


In [9]:
RANDOM_SEED = 42
OUTER_SPLITS = 5
INNER_SPLITS = 3

outer_splitter = StratifiedGroupKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

outer_folds = list(
    outer_splitter.split(
        X,
        y,
        groups
    )
)

outer_rows = []

for fold, (train_idx, test_idx) in enumerate(
    outer_folds,
    start=1
):
    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    outer_rows.append({
        "fold": fold,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_positive": int(y.iloc[train_idx].sum()),
        "train_control": int((y.iloc[train_idx] == 0).sum()),
        "test_positive": int(y.iloc[test_idx].sum()),
        "test_control": int((y.iloc[test_idx] == 0).sum()),
        "train_groups": len(train_groups),
        "test_groups": len(test_groups),
        "group_overlap": len(train_groups.intersection(test_groups)),
    })

outer_fold_summary = pd.DataFrame(outer_rows)

display(outer_fold_summary)

if outer_fold_summary["group_overlap"].sum() != 0:
    raise ValueError("Spatial leakage detected.")

outer_fold_summary.to_csv(
    OUTPUT_DIR / "outer_spatial_fold_summary.csv",
    index=False
)


,fold,train_rows,test_rows,train_positive,train_control,test_positive,test_control,train_groups,test_groups,group_overlap
0,1,464,118,154,310,40,78,132,35,0
1,2,463,119,154,309,40,79,134,33,0
2,3,467,115,156,311,38,77,135,32,0
3,4,467,115,156,311,38,77,132,35,0
4,5,467,115,156,311,38,77,135,32,0


# PART B — MODEL PIPELINES AND SEARCH SPACES

## 5. Logistic Regression

Scaling is performed inside the pipeline and therefore inside training folds only.


In [10]:
lr_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=RANDOM_SEED
        )
    ),
])

lr_params = {
    "model__C": [
        0.01,
        0.03,
        0.1,
        0.3,
        1.0,
        3.0,
        10.0,
        30.0,
        100.0,
    ],
    "model__penalty": [
        "l2",
    ],
    "model__solver": [
        "lbfgs",
    ],
}


## 6. Random Forest

In [11]:
rf_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
    ),
])

rf_params = {
    "model__n_estimators": [
        300,
        500,
        800,
        1200,
    ],
    "model__max_depth": [
        None,
        5,
        8,
        12,
        16,
    ],
    "model__min_samples_split": [
        2,
        5,
        10,
        20,
    ],
    "model__min_samples_leaf": [
        1,
        2,
        3,
        5,
        8,
    ],
    "model__max_features": [
        "sqrt",
        "log2",
        0.5,
        0.8,
    ],
}


## 7. XGBoost

In [12]:
xgb_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            tree_method="hist"
        )
    ),
])

xgb_params = {
    "model__n_estimators": [
        150,
        250,
        400,
        600,
        800,
    ],
    "model__max_depth": [
        2,
        3,
        4,
        5,
        6,
    ],
    "model__learning_rate": [
        0.01,
        0.03,
        0.05,
        0.1,
        0.2,
    ],
    "model__subsample": [
        0.6,
        0.8,
        1.0,
    ],
    "model__colsample_bytree": [
        0.6,
        0.8,
        1.0,
    ],
    "model__min_child_weight": [
        1,
        3,
        5,
        10,
    ],
    "model__reg_alpha": [
        0.0,
        0.01,
        0.1,
        1.0,
    ],
    "model__reg_lambda": [
        0.5,
        1.0,
        2.0,
        5.0,
        10.0,
    ],
    "model__scale_pos_weight": [
        1.0,
        1.5,
        2.0,
    ],
}


## 8. Benchmark model registry


In [13]:
MODEL_SPECS = {
    "Logistic Regression": {
        "pipeline": lr_pipeline,
        "params": lr_params,
        "n_iter": 9,
    },
    "Random Forest": {
        "pipeline": rf_pipeline,
        "params": rf_params,
        "n_iter": 24,
    },
    "XGBoost": {
        "pipeline": xgb_pipeline,
        "params": xgb_params,
        "n_iter": 30,
    },
}

print("Models:")
for name in MODEL_SPECS:
    print(" -", name)


Models:
 - Logistic Regression
 - Random Forest
 - XGBoost


# PART C — METRIC HELPERS

## 9. Compute classification metrics


In [14]:
METRIC_NAMES = [
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
]


def calculate_metrics(
    y_true,
    probability,
    threshold=0.5
):
    prediction = (
        np.asarray(probability)
        >= threshold
    ).astype(int)

    return {
        "accuracy":
            accuracy_score(
                y_true,
                prediction
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                prediction
            ),

        "precision":
            precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probability
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                probability
            ),
    }


# PART D — NESTED SPATIAL BENCHMARK

## 10. Train/tune/evaluate all three models

For every outer fold:

1. hold out one spatial test fold;
2. create an inner 3-fold spatial CV using only outer-training data;
3. tune hyperparameters using ROC-AUC;
4. refit the best model on the full outer-training data;
5. calculate training metrics;
6. calculate untouched outer-test metrics;
7. save out-of-fold probability predictions.

This gives both:
- generalization performance
- overfitting gaps


In [15]:
all_fold_results = []
best_parameter_rows = []
oof_rows = []

fitted_best_models = {}

for model_number, (model_name, spec) in enumerate(
    MODEL_SPECS.items(),
    start=1
):
    print("\n" + "=" * 70)
    print(f"MODEL {model_number}/3: {model_name}")
    print("=" * 70)

    fitted_best_models[model_name] = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_folds,
        start=1
    ):
        print(
            f"\n{model_name} | Outer fold {fold}/{OUTER_SPLITS}"
        )

        X_train = X.iloc[train_idx].copy()
        y_train = y.iloc[train_idx].copy()
        g_train = groups.iloc[train_idx].copy()

        X_test = X.iloc[test_idx].copy()
        y_test = y.iloc[test_idx].copy()

        inner_splitter = StratifiedGroupKFold(
            n_splits=INNER_SPLITS,
            shuffle=True,
            random_state=(
                RANDOM_SEED
                + fold
            )
        )

        search = RandomizedSearchCV(
            estimator=clone(
                spec["pipeline"]
            ),
            param_distributions=spec[
                "params"
            ],
            n_iter=spec[
                "n_iter"
            ],
            scoring="roc_auc",
            cv=inner_splitter,
            refit=True,
            random_state=(
                RANDOM_SEED
                + fold
            ),
            n_jobs=-1,
            verbose=0,
            return_train_score=True
        )

        start_time = time.time()

        search.fit(
            X_train,
            y_train,
            groups=g_train
        )

        elapsed = (
            time.time()
            - start_time
        )

        best_model = search.best_estimator_

        train_probability = (
            best_model.predict_proba(
                X_train
            )[:, 1]
        )

        test_probability = (
            best_model.predict_proba(
                X_test
            )[:, 1]
        )

        train_metrics = calculate_metrics(
            y_train,
            train_probability,
            threshold=0.5
        )

        test_metrics = calculate_metrics(
            y_test,
            test_probability,
            threshold=0.5
        )

        result_row = {
            "model":
                model_name,

            "outer_fold":
                fold,

            "inner_best_roc_auc":
                search.best_score_,

            "fit_seconds":
                elapsed,
        }

        for metric in METRIC_NAMES:
            result_row[
                f"train_{metric}"
            ] = train_metrics[
                metric
            ]

            result_row[
                f"test_{metric}"
            ] = test_metrics[
                metric
            ]

            result_row[
                f"gap_{metric}"
            ] = (
                train_metrics[
                    metric
                ]
                - test_metrics[
                    metric
                ]
            )

        all_fold_results.append(
            result_row
        )

        best_parameter_rows.append({
            "model":
                model_name,

            "outer_fold":
                fold,

            "inner_best_roc_auc":
                search.best_score_,

            "best_params":
                str(
                    search.best_params_
                ),
        })

        for local_position, original_idx in enumerate(
            test_idx
        ):
            oof_rows.append({
                "model":
                    model_name,

                "outer_fold":
                    fold,

                "row_index":
                    int(
                        original_idx
                    ),

                "sample_id":
                    df.iloc[
                        original_idx
                    ][
                        "sample_id"
                    ],

                "target":
                    int(
                        y.iloc[
                            original_idx
                        ]
                    ),

                "probability":
                    float(
                        test_probability[
                            local_position
                        ]
                    ),

                "prediction_0_5":
                    int(
                        test_probability[
                            local_position
                        ]
                        >= 0.5
                    ),

                "spatial_block_id":
                    groups.iloc[
                        original_idx
                    ],
            })

        fitted_best_models[
            model_name
        ].append(
            best_model
        )

        print(
            "  Inner best ROC-AUC:",
            round(
                search.best_score_,
                3
            )
        )

        print(
            "  Train ROC-AUC:",
            round(
                train_metrics[
                    "roc_auc"
                ],
                3
            )
        )

        print(
            "  Test ROC-AUC:",
            round(
                test_metrics[
                    "roc_auc"
                ],
                3
            )
        )

        print(
            "  Test balanced accuracy:",
            round(
                test_metrics[
                    "balanced_accuracy"
                ],
                3
            )
        )

        print(
            "  Test recall:",
            round(
                test_metrics[
                    "recall"
                ],
                3
            )
        )


fold_results = pd.DataFrame(
    all_fold_results
)

best_parameters = pd.DataFrame(
    best_parameter_rows
)

oof_predictions = pd.DataFrame(
    oof_rows
)

display(
    fold_results
)

fold_results.to_csv(
    OUTPUT_DIR / "nested_spatial_fold_results.csv",
    index=False
)

best_parameters.to_csv(
    OUTPUT_DIR / "best_hyperparameters_by_fold.csv",
    index=False
)

oof_predictions.to_csv(
    OUTPUT_DIR / "nested_oof_predictions.csv",
    index=False
)



MODEL 1/3: Logistic Regression

Logistic Regression | Outer fold 1/5
  Inner best ROC-AUC: 0.428
  Train ROC-AUC: 0.651
  Test ROC-AUC: 0.777
  Test balanced accuracy: 0.702
  Test recall: 0.725

Logistic Regression | Outer fold 2/5
  Inner best ROC-AUC: 0.6
  Train ROC-AUC: 0.71
  Test ROC-AUC: 0.324
  Test balanced accuracy: 0.36
  Test recall: 0.125

Logistic Regression | Outer fold 3/5
  Inner best ROC-AUC: 0.411
  Train ROC-AUC: 0.664
  Test ROC-AUC: 0.753
  Test balanced accuracy: 0.68
  Test recall: 0.684

Logistic Regression | Outer fold 4/5
  Inner best ROC-AUC: 0.713
  Train ROC-AUC: 0.753
  Test ROC-AUC: 0.403
  Test balanced accuracy: 0.457
  Test recall: 0.395

Logistic Regression | Outer fold 5/5
  Inner best ROC-AUC: 0.536
  Train ROC-AUC: 0.707
  Test ROC-AUC: 0.537
  Test balanced accuracy: 0.515
  Test recall: 0.368

MODEL 2/3: Random Forest

Random Forest | Outer fold 1/5
  Inner best ROC-AUC: 0.68
  Train ROC-AUC: 0.989
  Test ROC-AUC: 0.799
  Test balanced accurac

,model,outer_fold,inner_best_roc_auc,fit_seconds,train_accuracy,test_accuracy,gap_accuracy,train_balanced_accuracy,test_balanced_accuracy,gap_balanced_accuracy,...,gap_recall,train_f1,test_f1,gap_f1,train_roc_auc,test_roc_auc,gap_roc_auc,train_pr_auc,test_pr_auc,gap_pr_auc
0,Logistic Regression,1,0.428138,17.348775,0.605603,0.694915,-0.089312,0.595371,0.702244,-0.106873,...,-0.160065,0.487395,0.617021,-0.129626,0.651320,0.776923,-0.125603,0.530875,0.660581,-0.129706
1,Logistic Regression,2,0.600242,0.435590,0.680346,0.436975,0.243371,0.662800,0.359968,0.302832,...,0.485390,0.559524,0.129870,0.429654,0.710167,0.323734,0.386433,0.574291,0.257331,0.316960
2,Logistic Regression,3,0.411134,0.310096,0.640257,0.678261,-0.038004,0.640449,0.679768,-0.039319,...,-0.043185,0.543478,0.584270,-0.040791,0.663575,0.752563,-0.088988,0.514376,0.618691,-0.104316
3,Logistic Regression,4,0.713340,0.397719,0.672377,0.478261,0.194116,0.672551,0.457109,0.215443,...,0.278340,0.578512,0.333333,0.245179,0.753071,0.403281,0.349790,0.620955,0.345617,0.275338
4,Logistic Regression,5,0.536405,0.319855,0.659529,0.565217,0.094312,0.666100,0.515379,0.150720,...,0.317476,0.573727,0.358974,0.214752,0.706921,0.536910,0.170011,0.608082,0.348358,0.259724
5,Random Forest,1,0.680342,53.826374,0.941810,0.745763,0.196048,0.940113,0.710256,0.229857,...,0.335065,0.914286,0.615385,0.298901,0.989338,0.799038,0.190300,0.979887,0.691018,0.288869
6,Random Forest,2,0.688231,59.268647,0.831533,0.588235,0.243298,0.823299,0.498576,0.324723,...,0.573701,0.759259,0.268657,0.490603,0.935464,0.524367,0.411097,0.898113,0.348803,0.549310
7,Random Forest,3,0.597005,50.271412,0.916488,0.773913,0.142575,0.919728,0.744532,0.175196,...,0.271592,0.881459,0.657895,0.223564,0.979759,0.807245,0.172514,0.963472,0.743808,0.219664
8,Random Forest,4,0.791996,59.891737,0.884368,0.669565,0.214803,0.873248,0.606630,0.266618,...,0.418691,0.829114,0.457143,0.371971,0.957684,0.637389,0.320295,0.940653,0.519001,0.421652
9,Random Forest,5,0.749953,52.112629,0.942184,0.765217,0.176967,0.937423,0.684723,0.252700,...,0.475709,0.914286,0.557377,0.356909,0.990416,0.762474,0.227941,0.983048,0.667465,0.315583


# PART E — FINAL PERFORMANCE SUMMARY

## 11. Outer-test mean ± standard deviation


In [16]:
summary_rows = []

for model_name, group in fold_results.groupby(
    "model"
):
    row = {
        "model":
            model_name
    }

    for metric in METRIC_NAMES:

        row[
            f"{metric}_mean"
        ] = group[
            f"test_{metric}"
        ].mean()

        row[
            f"{metric}_std"
        ] = group[
            f"test_{metric}"
        ].std()

    row[
        "roc_auc_train_mean"
    ] = group[
        "train_roc_auc"
    ].mean()

    row[
        "roc_auc_test_mean"
    ] = group[
        "test_roc_auc"
    ].mean()

    row[
        "roc_auc_overfit_gap_mean"
    ] = group[
        "gap_roc_auc"
    ].mean()

    row[
        "f1_train_mean"
    ] = group[
        "train_f1"
    ].mean()

    row[
        "f1_test_mean"
    ] = group[
        "test_f1"
    ].mean()

    row[
        "f1_overfit_gap_mean"
    ] = group[
        "gap_f1"
    ].mean()

    summary_rows.append(
        row
    )

benchmark_summary = pd.DataFrame(
    summary_rows
).sort_values(
    "roc_auc_mean",
    ascending=False
)

display(
    benchmark_summary
)

benchmark_summary.to_csv(
    OUTPUT_DIR / "final_model_benchmark_summary.csv",
    index=False
)


,model,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,...,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,roc_auc_train_mean,roc_auc_test_mean,roc_auc_overfit_gap_mean,f1_train_mean,f1_test_mean,f1_overfit_gap_mean
2,XGBoost,0.718402,0.057493,0.626802,0.102021,0.544681,0.322614,0.351842,0.250567,0.409584,...,0.709606,0.128446,0.591218,0.167395,0.973288,0.709606,0.263683,0.863949,0.409584,0.454364
1,Random Forest,0.708539,0.078841,0.648944,0.098206,0.572387,0.158907,0.470263,0.169659,0.511291,...,0.706103,0.122264,0.594019,0.160467,0.970532,0.706103,0.264429,0.859681,0.511291,0.348390
0,Logistic Regression,0.570726,0.115603,0.542894,0.146377,0.364088,0.165440,0.459474,0.247672,0.404694,...,0.558682,0.203127,0.446116,0.181021,0.697011,0.558682,0.138328,0.548527,0.404694,0.143833


## 12. Overall out-of-fold metrics

Each row in these metrics was predicted while belonging to an untouched outer spatial fold.


In [17]:
overall_rows = []
confusion_rows = []

for model_name, group in oof_predictions.groupby(
    "model"
):
    group = group.sort_values(
        "row_index"
    )

    y_true = group[
        "target"
    ].to_numpy()

    probability = group[
        "probability"
    ].to_numpy()

    metrics = calculate_metrics(
        y_true,
        probability,
        threshold=0.5
    )

    overall_rows.append({
        "model":
            model_name,

        **metrics
    })

    cm = confusion_matrix(
        y_true,
        (
            probability >= 0.5
        ).astype(int)
    )

    tn, fp, fn, tp = cm.ravel()

    confusion_rows.append({
        "model":
            model_name,

        "true_negative":
            int(tn),

        "false_positive":
            int(fp),

        "false_negative":
            int(fn),

        "true_positive":
            int(tp),
    })


overall_oof_metrics = pd.DataFrame(
    overall_rows
).sort_values(
    "roc_auc",
    ascending=False
)

confusion_summary = pd.DataFrame(
    confusion_rows
)


display(
    overall_oof_metrics
)

display(
    confusion_summary
)


overall_oof_metrics.to_csv(
    OUTPUT_DIR / "final_model_overall_oof_metrics.csv",
    index=False
)

confusion_summary.to_csv(
    OUTPUT_DIR / "final_model_confusion_matrices.csv",
    index=False
)


,model,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
1,Random Forest,0.707904,0.648196,0.575949,0.469072,0.517045,0.711048,0.585313
2,XGBoost,0.718213,0.626289,0.641509,0.350515,0.453333,0.698108,0.560233
0,Logistic Regression,0.570447,0.542526,0.380342,0.458763,0.415888,0.544399,0.396906


,model,true_negative,false_positive,false_negative,true_positive
0,Logistic Regression,243,145,105,89
1,Random Forest,321,67,103,91
2,XGBoost,350,38,126,68


# PART F — LITERATURE BENCHMARK TABLE

## 13. Published studies

These values are contextual literature benchmarks.

They are **not acceptance thresholds** because:
- study areas differ;
- conditioning factors differ;
- sample sizes differ;
- pseudo-absence/non-landslide sampling differs;
- validation protocols differ.

Your spatial outer CV is intentionally stricter than a simple random split.


In [18]:
literature_benchmarks = pd.DataFrame([
    {
        "study":
            "Wang et al. (2024)",

        "model":
            "Random Forest",

        "accuracy":
            0.848,

        "f1":
            0.740,

        "roc_auc":
            0.904,

        "validation_note":
            "Random 80/20 split; 19 conditioning factors",
    },

    {
        "study":
            "Zhang et al. (2024)",

        "model":
            "Logistic Regression",

        "accuracy":
            np.nan,

        "f1":
            np.nan,

        "roc_auc":
            0.752,

        "validation_note":
            "Different study area and factor-selection workflow",
    },

    {
        "study":
            "Zhang et al. (2024)",

        "model":
            "Random Forest",

        "accuracy":
            np.nan,

        "f1":
            np.nan,

        "roc_auc":
            0.965,

        "validation_note":
            "Different study area and factor-selection workflow",
    },

    {
        "study":
            "Zhang et al. (2024)",

        "model":
            "XGBoost",

        "accuracy":
            np.nan,

        "f1":
            np.nan,

        "roc_auc":
            0.998,

        "validation_note":
            "Different study area and factor-selection workflow",
    },

    {
        "study":
            "Halder et al. (2025)",

        "model":
            "Logistic Regression",

        "accuracy":
            0.860,

        "f1":
            0.880,

        "roc_auc":
            0.935,

        "validation_note":
            "Random 70/30 split; 21 conditioning factors; RFECV/tuning",
    },

    {
        "study":
            "Halder et al. (2025)",

        "model":
            "Random Forest",

        "accuracy":
            0.943,

        "f1":
            0.949,

        "roc_auc":
            0.983,

        "validation_note":
            "Random 70/30 split; 21 conditioning factors; RFECV/tuning",
    },

    {
        "study":
            "Halder et al. (2025)",

        "model":
            "XGBoost",

        "accuracy":
            0.943,

        "f1":
            np.nan,

        "roc_auc":
            0.987,

        "validation_note":
            "Random 70/30 split; 21 conditioning factors; RFECV/tuning",
    },
])

display(
    literature_benchmarks
)

literature_benchmarks.to_csv(
    OUTPUT_DIR / "published_literature_benchmarks.csv",
    index=False
)


,study,model,accuracy,f1,roc_auc,validation_note
0,Wang et al. (2024),Random Forest,0.848,0.740,0.904,Random 80/20 split; 19 conditioning factors
1,Zhang et al. (2024),Logistic Regression,NaN,NaN,0.752,Different study area and factor-selection work...
2,Zhang et al. (2024),Random Forest,NaN,NaN,0.965,Different study area and factor-selection work...
3,Zhang et al. (2024),XGBoost,NaN,NaN,0.998,Different study area and factor-selection work...
4,Halder et al. (2025),Logistic Regression,0.860,0.880,0.935,Random 70/30 split; 21 conditioning factors; R...
5,Halder et al. (2025),Random Forest,0.943,0.949,0.983,Random 70/30 split; 21 conditioning factors; R...
6,Halder et al. (2025),XGBoost,0.943,NaN,0.987,Random 70/30 split; 21 conditioning factors; R...


# PART G — PRACTICAL PERFORMANCE GUIDE

## 14. Interpretation bands for this FYP

These are **project heuristics**, not universal scientific cutoffs.

### ROC-AUC

- 0.50 → no discrimination
- 0.60–0.70 → weak
- 0.70–0.80 → acceptable/moderate
- 0.80–0.90 → strong
- >0.90 → excellent, but inspect carefully for leakage/overfitting

### Balanced Accuracy

- 0.50 → chance
- 0.60–0.70 → limited/moderate
- 0.70–0.80 → useful
- >0.80 → strong

### Recall

For a landslide-screening system, missing positives is costly.

A practical goal is:
- <0.50 → weak for screening
- 0.50–0.70 → moderate
- >=0.70 → desirable

This is not a formal universal threshold.

### F1

- <0.50 → weak
- 0.50–0.60 → modest
- 0.60–0.70 → useful
- >0.70 → strong

### PR-AUC

No-skill baseline for this dataset is around the positive prevalence (~0.333).

Therefore:
- ~0.333 → little/no gain over prevalence baseline
- >0.50 → meaningful improvement
- >0.65 → strong for this project

### Fold variability

Approximate interpretation:
- std <0.05 → stable
- 0.05–0.10 → moderate variation
- >0.10 → spatial instability deserves attention

### Train-test ROC-AUC gap

- <0.05 → small
- 0.05–0.10 → moderate
- >0.10 → possible overfitting

Again, these are diagnostic heuristics, not pass/fail laws.


In [19]:
metric_interpretation = pd.DataFrame([
    {
        "metric":
            "ROC-AUC",

        "primary_goal_for_this_FYP":
            "Prefer >=0.75; >=0.80 would be strong",

        "note":
            "Threshold-independent; primary ranking metric"
    },

    {
        "metric":
            "Balanced Accuracy",

        "primary_goal_for_this_FYP":
            "Prefer >=0.70",

        "note":
            "More informative than raw accuracy for 1:2 classes"
    },

    {
        "metric":
            "Recall",

        "primary_goal_for_this_FYP":
            "Prefer >=0.65-0.70",

        "note":
            "Important because false negatives are missed landslide locations"
    },

    {
        "metric":
            "F1",

        "primary_goal_for_this_FYP":
            "Prefer >=0.60; >=0.70 strong",

        "note":
            "Balances precision and recall"
    },

    {
        "metric":
            "PR-AUC",

        "primary_goal_for_this_FYP":
            "Clearly >0.333; prefer >0.50",

        "note":
            "No-skill baseline approximately equals positive prevalence"
    },

    {
        "metric":
            "Accuracy",

        "primary_goal_for_this_FYP":
            "Must be interpreted against ~0.667 majority baseline",

        "note":
            "Do not use alone"
    },

    {
        "metric":
            "Fold ROC-AUC std",

        "primary_goal_for_this_FYP":
            "Prefer <0.10",

        "note":
            "Lower indicates better spatial stability"
    },

    {
        "metric":
            "Train-test ROC-AUC gap",

        "primary_goal_for_this_FYP":
            "Prefer <0.10",

        "note":
            "Large gap suggests overfitting"
    },
])

display(
    metric_interpretation
)

metric_interpretation.to_csv(
    OUTPUT_DIR / "metric_interpretation_guide.csv",
    index=False
)


,metric,primary_goal_for_this_FYP,note
0,ROC-AUC,Prefer >=0.75; >=0.80 would be strong,Threshold-independent; primary ranking metric
1,Balanced Accuracy,Prefer >=0.70,More informative than raw accuracy for 1:2 cla...
2,Recall,Prefer >=0.65-0.70,Important because false negatives are missed l...
3,F1,Prefer >=0.60; >=0.70 strong,Balances precision and recall
4,PR-AUC,Clearly >0.333; prefer >0.50,No-skill baseline approximately equals positiv...
5,Accuracy,Must be interpreted against ~0.667 majority ba...,Do not use alone
6,Fold ROC-AUC std,Prefer <0.10,Lower indicates better spatial stability
7,Train-test ROC-AUC gap,Prefer <0.10,Large gap suggests overfitting


# PART H — MODEL SELECTION RULE

## 15. Recommended decision hierarchy

Do not automatically choose the model with the highest training score.

Recommended order:

1. Compare **outer-test ROC-AUC**
2. Check **outer-test PR-AUC**
3. Check **Balanced Accuracy**
4. Check **Recall**
5. Check **F1**
6. Inspect fold standard deviations
7. Inspect train-test gaps
8. Inspect confusion matrices
9. Consider interpretability and map suitability

A model with ROC-AUC 0.80 and stable folds may be preferable to a model with ROC-AUC 0.84 but severe spatial instability or overfitting.


# Outputs to send for review

Please send:

```text
outer_spatial_fold_summary.csv
nested_spatial_fold_results.csv
final_model_benchmark_summary.csv
final_model_overall_oof_metrics.csv
final_model_confusion_matrices.csv
best_hyperparameters_by_fold.csv
```

Then we can make the final LR vs RF vs XGBoost comparison and decide whether:

- the static feature set is sufficient;
- additional geology / land-cover / waterway predictors should be added;
- XGBoost is genuinely better;
- the strongest model is ready for Malaysia-wide susceptibility mapping.
